In [1]:
import random
import gymnasium as gym # contains the cartpole-v1 pendulum environment
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

import time
from collections import deque

In [2]:
environment=gym.make('CartPole-v1',render_mode='human') # create the environment

In [3]:
#Some playing around
state, info = environment.reset()
state # array  of cart_position, cart_velocity, pole_angle, pole_angular_velocity
number_of_actions= environment.action_space.n
print(number_of_actions) # 2 left or right 0 or 1

2


In [4]:
# Build the Q-network instead of a huge Q-table(main point of deep-Q learning use a neural network as the approximation of the Q function)
class DeepQ(nn.Module):
  def __init__(self, state_size, number_of_actions):
    super().__init__()
    self.layers= nn.Sequential(
        nn.Linear(state_size, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, number_of_actions)
    )

  def forward(self, state):
    return self.layers(state)


In [5]:
device='cpu'
policy_model= DeepQ(4, 2).to(device) # state_size=4, no of actions=2
target_model= DeepQ(4, 2).to(device)
optimizer= optim.Adam(policy_model.parameters(), lr=0.001)

In [6]:
loss_function = nn.MSELoss() #mean square error as loss

In [7]:
replay_memory= deque(maxlen=10000) # store the experiences

In [8]:
# Choosing actions intially by greedy then by epsilon greedy
def choose_action(state):
  if random.random() < epsilon :
    return environment.action_space.sample()

  state_tensor= torch.FloatTensor(state).unsqueeze(0).to(device)

  with torch.no_grad():
    q_values = policy_model(state_tensor)

  action= torch.argmax(q_values).detach().item() #whichever action has higher value

  return action

In [9]:
batch_size = 64
gamma = 0.99
epsilon=1

def train_network(train_steps):

  batch = random.sample(replay_memory, batch_size) # instead of only the last experience, sample last 64 randomly
  states, actions,rewards, next_states, dones = zip(*batch) # separate each column into arrays

  # convert all to pytorch tensors to work with them
  states = torch.as_tensor(np.array(states), dtype=torch.float32, device=device)
  next_states = torch.as_tensor(np.array(next_states), dtype=torch.float32, device=device)
    
  actions = torch.as_tensor(actions, dtype=torch.long, device=device)
  rewards = torch.as_tensor(rewards, dtype=torch.float32, device=device)
  dones = torch.as_tensor(dones, dtype=torch.float32, device=device)

  all_q_values = policy_model(states) # ask model for all q values of every action in every state

  predicted_q_values = all_q_values.gather(dim=1, index=actions.unsqueeze(1)).squeeze(1) # keep only the q value of the action that was taken

  # Compute the target from Bellman Expectation

  with torch.no_grad():
    future_q_values= target_model(next_states) # ask model for all q values of every action in every possible future state
    maximum_future_q=future_q_values.max(dim=1)[0]

    target_q_values=(rewards + gamma*maximum_future_q*(1-dones)) #Bellman

  loss= loss_function(predicted_q_values, target_q_values) # compute diff between the 2 q values

  train_steps+=1
  if(train_steps%500 ==0):
      target_model.load_state_dict(policy_model.state_dict())
  # backprop and train the model
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

  return train_steps

In [10]:
train_steps=0
for episode in range(200):
    state, info = environment.reset()
    done = False

    while not done:
        action = choose_action(state)
        next_state, reward, terminated, truncated, info = environment.step(action)
        done = terminated or truncated
        # time.sleep(0.02)
        replay_memory.append(
            (
                state,
                action,
                reward,
                next_state,
                done
            )
        )
        state = next_state
        if len(replay_memory) >= batch_size: # don't train till sufficient no of experiences present
            train_steps=train_network(train_steps)

    print(f"{episode} {train_steps}")
    epsilon = max(
        0.01,
        epsilon * 0.995
    )
environment.close()

0 0
1 0
2 0
3 4
4 22
5 45
6 55
7 75
8 88
9 107
10 119
11 130
12 144
13 158
14 168
15 184
16 199
17 217
18 242
19 267
20 288
21 309
22 329
23 346
24 389
25 404
26 460
27 474
28 496
29 507
30 521
31 533
32 558
33 581
34 599
35 609
36 623
37 639
38 649
39 667
40 676
41 686
42 730
43 739
44 767
45 788
46 798
47 816
48 828
49 851
50 864
51 875
52 906
53 925
54 940
55 959
56 970
57 990
58 1003
59 1020
60 1033
61 1074
62 1097
63 1115
64 1153
65 1176
66 1199
67 1267
68 1290
69 1316
70 1343
71 1354
72 1382
73 1396
74 1461
75 1480
76 1490
77 1519
78 1542
79 1557
80 1588
81 1682
82 1696
83 1792
84 1855
85 1868
86 1920
87 1946
88 1984
89 1997
90 2011
91 2023
92 2057
93 2104
94 2133
95 2156
96 2171
97 2186
98 2227
99 2243
100 2280
101 2315
102 2328
103 2392
104 2429
105 2447
106 2582
107 2616
108 2652
109 2689
110 2717
111 2755
112 2803
113 2833
114 2863
115 2881
116 2998
117 3044
118 3102
119 3169
120 3229
121 3268
122 3315
123 3477
124 3506
125 3699
126 3739
127 3788
128 3854
129 4015
130 4072
13